In [1]:
import os
import sys
import gc
import re
import json
import math
import time
import random
import inspect
import platform
import hashlib
import importlib.metadata as importlib_metadata
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

try:
    from peft import LoraConfig, PeftModel, get_peft_model
except ImportError as exc:
    raise ImportError("Notebook này cần package peft để train LoRA. Hãy bật môi trường Kaggle có sẵn peft hoặc thêm peft vào dataset package offline.") from exc

try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

pd.set_option("display.max_colwidth", 180)
print("Python:", sys.version.replace("\n", " "))
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for gpu_idx in range(torch.cuda.device_count()):
        print("GPU", gpu_idx, torch.cuda.get_device_name(gpu_idx), torch.cuda.get_device_capability(gpu_idx))

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True | GPU count: 2
GPU 0 Tesla T4 (7, 5)
GPU 1 Tesla T4 (7, 5)


In [2]:
IS_KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

KAGGLE_INPUT_ROOT = Path("/kaggle/input/datasets/phamanhtuanas/gpt2-math/preprocessing_experiments/preprocessing_experiments")
MODEL_LOAD_PATH = None
BASE_MODEL_NAME = "NlpHUST/gpt2-vietnamese"
OUTPUT_ROOT = Path("/kaggle/working/variant_experiments") if IS_KAGGLE else PROJECT_ROOT / "outputs" / "variant_experiments"

RUN_MODE = "light_ablation"
VARIANTS_TO_RUN = ["v04_decimal_normalized", "v05_strip_asy", "v06_keep_asy_if_short"]
FORCE_RERUN = False
ALLOW_CPU_TRAINING = False
RUN_TEST_INFERENCE = False
SUBMISSION_RUN_ID = None
TEST_FILE = None
SEED = 42

PROMPT_TEMPLATE = "Câu hỏi: {query}\nLời giải:\n{response}"
INFERENCE_PREFIX = "Câu hỏi: {query}\nLời giải:\n"
TEMPLATE_NAME = "basic_vi_math_solution"
MASK_PROMPT_LOSS = True
SAMPLING_STRATEGY = "stratified_by_type"

MODE_CONFIGS = {
    "sanity_check": {
        "train_sample_count": 96,
        "valid_random_sample_count": 24,
        "valid_grouped_sample_count": 24,
        "variant_limit": 1,
        "epochs": 1,
        "max_steps": 5,
        "max_length": 256,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 1,
        "learning_rate": 2e-4,
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "logging_steps": 1,
        "trainer_eval_sample_count": 24,
        "generation_batch_size": 2,
        "max_new_tokens": 96,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
    },
    "light_ablation": {
        "train_sample_count": 10000,
        "valid_random_sample_count": 1000,
        "valid_grouped_sample_count": 1000,
        "variant_limit": None,
        "epochs": 1,
        "max_steps": None,
        "max_length": 512,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "logging_steps": 25,
        "trainer_eval_sample_count": 256,
        "generation_batch_size": 4,
        "max_new_tokens": 256,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    "medium_ablation": {
        "train_sample_count": 50000,
        "valid_random_sample_count": None,
        "valid_grouped_sample_count": None,
        "variant_limit": 3,
        "epochs": 1,
        "max_steps": None,
        "max_length": 512,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "logging_steps": 50,
        "trainer_eval_sample_count": 512,
        "generation_batch_size": 4,
        "max_new_tokens": 256,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
    },
    "strong_final_train": {
        "train_sample_count": None,
        "valid_random_sample_count": None,
        "valid_grouped_sample_count": None,
        "variant_limit": 1,
        "epochs": 2,
        "max_steps": None,
        "max_length": 768,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "learning_rate": 1e-4,
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "logging_steps": 50,
        "trainer_eval_sample_count": 1024,
        "generation_batch_size": 2,
        "max_new_tokens": 320,
        "lora_r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.05,
    },
}

GENERATION_CONFIG = {
    "do_sample": False,
    "num_beams": 1,
    "temperature": None,
    "top_p": None,
    "repetition_penalty": 1.05,
    "no_repeat_ngram_size": 4,
}

LORA_TARGET_MODULES = ["c_attn", "c_proj"]
RUN_MODE_PREFIX = {
    "sanity_check": "sanity",
    "light_ablation": "light",
    "medium_ablation": "medium",
    "strong_final_train": "strong",
}

if RUN_MODE not in MODE_CONFIGS:
    raise ValueError(f"RUN_MODE không hợp lệ: {RUN_MODE}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("RUN_MODE:", RUN_MODE)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

RUN_MODE: light_ablation
OUTPUT_ROOT: /kaggle/working/variant_experiments


In [3]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


def json_default(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=json_default)


def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False, default=json_default) + "\n")


def load_json_or_jsonl(path):
    path = Path(path)
    with path.open("r", encoding="utf-8-sig") as f:
        first = f.read(1)
        f.seek(0)
        if first == "[":
            return json.load(f)
        return [json.loads(line) for line in f if line.strip()]


def load_json_optional(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def stable_int(value):
    return int(hashlib.blake2b(str(value).encode("utf-8"), digest_size=8).hexdigest(), 16)


seed_everything(SEED)

use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
use_fp16 = bool(torch.cuda.is_available() and not use_bf16)
MIXED_PRECISION = "bf16" if use_bf16 else "fp16" if use_fp16 else "none"

environment_report = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "transformers_version": package_version("transformers"),
    "datasets_version": package_version("datasets"),
    "peft_version": package_version("peft"),
    "accelerate_version": package_version("accelerate"),
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_count": torch.cuda.device_count(),
    "mixed_precision": MIXED_PRECISION,
}

global_config = {
    "run_mode": RUN_MODE,
    "variants_to_run": VARIANTS_TO_RUN,
    "seed": SEED,
    "prompt": {
        "template_name": TEMPLATE_NAME,
        "template": PROMPT_TEMPLATE,
        "inference_prefix": INFERENCE_PREFIX,
        "mask_prompt_loss": MASK_PROMPT_LOSS,
    },
    "mode_configs": MODE_CONFIGS,
    "generation_config": GENERATION_CONFIG,
    "lora_target_modules": LORA_TARGET_MODULES,
    "sampling_strategy": SAMPLING_STRATEGY,
}

save_json(environment_report, OUTPUT_ROOT / "environment_report.json")
save_json(global_config, OUTPUT_ROOT / "global_config.json")
print(json.dumps(environment_report, ensure_ascii=False, indent=2))

{
  "python_version": "3.12.12",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch_version": "2.10.0+cu128",
  "transformers_version": "5.0.0",
  "datasets_version": "4.8.3",
  "peft_version": "0.18.1",
  "accelerate_version": "1.12.0",
  "cuda_available": true,
  "gpu_name": "Tesla T4",
  "gpu_count": 2,
  "mixed_precision": "bf16"
}


In [4]:
REQUIRED_VARIANT_FILES = [
    "train.jsonl",
    "valid_random.jsonl",
    "valid_grouped.jsonl",
    "metrics_pretrain.json",
    "variant_config.json",
]


def is_variant_dir(path):
    path = Path(path)
    return path.is_dir() and all((path / name).exists() for name in REQUIRED_VARIANT_FILES)


def looks_like_variant_root(path):
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False
    return any(is_variant_dir(child) for child in path.iterdir() if child.is_dir())


def candidate_input_roots():
    candidates = []
    if KAGGLE_INPUT_ROOT:
        candidates.append(Path(KAGGLE_INPUT_ROOT))
    if IS_KAGGLE:
        base = Path("/kaggle/input")
        candidates.append(base)
        candidates.extend(sorted([p for p in base.glob("*") if p.is_dir()]))
        candidates.extend(sorted([p for p in base.glob("*/*") if p.is_dir()]))
        candidates.extend(sorted([p for p in base.glob("*/*/*") if p.is_dir()]))
    candidates.extend([
        PROJECT_ROOT / "data" / "variants",
        PROJECT_ROOT / "outputs" / "variants",
        PROJECT_ROOT / "data",
        PROJECT_ROOT,
    ])
    deduped = []
    seen = set()
    for item in candidates:
        key = str(item.resolve()) if item.exists() else str(item)
        if key not in seen:
            seen.add(key)
            deduped.append(item)
    return deduped


def find_input_root():
    for cand in candidate_input_roots():
        if looks_like_variant_root(cand):
            return cand
    checked = "\n".join(str(x) for x in candidate_input_roots())
    raise FileNotFoundError("Không tìm thấy folder chứa các variant. Các path đã kiểm tra:\n" + checked)


def discover_variants(input_root):
    variants = [child for child in Path(input_root).iterdir() if is_variant_dir(child)]
    variants = sorted(variants, key=lambda p: p.name)
    if not variants:
        raise FileNotFoundError(f"Không có variant hợp lệ trong {input_root}")
    return variants


def resolve_model_path():
    candidates = []
    if MODEL_LOAD_PATH:
        candidates.append(Path(MODEL_LOAD_PATH))
    if IS_KAGGLE:
        candidates.extend([
            Path("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese"),
            Path("/kaggle/input/nlphustgpt2-vietnamese"),
            Path("/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese"),
            Path("/kaggle/input/nlphustgpt2-vietnamese/nlphust-gpt2-vietnamese"),
        ])
    candidates.extend([
        PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese",
        PROJECT_ROOT / "models" / "gpt2-vietnamese",
    ])
    for cand in candidates:
        if cand.exists():
            return str(cand), True
    return BASE_MODEL_NAME, False


INPUT_ROOT = find_input_root()
MODEL_NAME_OR_PATH, LOCAL_FILES_ONLY = resolve_model_path()
ALL_VARIANT_DIRS = discover_variants(INPUT_ROOT)

selected_names = list(VARIANTS_TO_RUN) if VARIANTS_TO_RUN else [p.name for p in ALL_VARIANT_DIRS]
variant_map = {p.name: p for p in ALL_VARIANT_DIRS}
missing_variants = [name for name in selected_names if name not in variant_map]
if missing_variants:
    raise ValueError("VARIANTS_TO_RUN không tồn tại: " + ", ".join(missing_variants))

mode_limit = MODE_CONFIGS[RUN_MODE].get("variant_limit")
if mode_limit is not None:
    selected_names = selected_names[: int(mode_limit)]

VARIANT_DIRS = [variant_map[name] for name in selected_names]

print("INPUT_ROOT:", INPUT_ROOT)
print("MODEL_NAME_OR_PATH:", MODEL_NAME_OR_PATH)
print("LOCAL_FILES_ONLY:", LOCAL_FILES_ONLY)
print("ALL_VARIANTS:", [p.name for p in ALL_VARIANT_DIRS])
print("RUN_VARIANTS:", [p.name for p in VARIANT_DIRS])

INPUT_ROOT: /kaggle/input/datasets/phamanhtuanas/gpt2-math/preprocessing_experiments/preprocessing_experiments
MODEL_NAME_OR_PATH: NlpHUST/gpt2-vietnamese
LOCAL_FILES_ONLY: False
ALL_VARIANTS: ['v00_minimal', 'v01_anchor_normalized', 'v02_anchor_dedup_conflict', 'v03_latex_artifact_fixed', 'v04_decimal_normalized', 'v05_strip_asy', 'v06_keep_asy_if_short']
RUN_VARIANTS: ['v04_decimal_normalized', 'v05_strip_asy', 'v06_keep_asy_if_short']


In [5]:
REQUIRED_FIELDS = [
    "id",
    "raw_index",
    "query",
    "response",
    "final_answer",
    "type",
    "source_group",
    "aug_type",
    "original_question_hash",
    "preprocess_variant",
]

FORBIDDEN_FIELDS = ["input_ids", "labels", "attention_mask"]


def validate_records(records, variant_name, split_name):
    errors = []
    warnings = []
    for idx, rec in enumerate(records):
        missing = [field for field in REQUIRED_FIELDS if field not in rec]
        forbidden = [field for field in FORBIDDEN_FIELDS if field in rec]
        if missing:
            errors.append({"row": idx, "error": "missing_fields", "fields": missing})
        if forbidden:
            errors.append({"row": idx, "error": "forbidden_fields", "fields": forbidden})
        if not str(rec.get("query", "")).strip():
            errors.append({"row": idx, "error": "empty_query"})
        if split_name in {"train", "valid_random", "valid_grouped"} and not str(rec.get("response", "")).strip():
            errors.append({"row": idx, "error": "empty_response"})
        if split_name in {"valid_random", "valid_grouped"} and not str(rec.get("final_answer", "")).strip():
            errors.append({"row": idx, "error": "empty_final_answer"})
        if rec.get("preprocess_variant") != variant_name:
            errors.append({"row": idx, "error": "variant_mismatch", "value": rec.get("preprocess_variant")})
        if len(errors) >= 30:
            break
    type_counter = Counter(str(rec.get("type", "unknown")) for rec in records)
    report = {
        "variant": variant_name,
        "split": split_name,
        "num_records": len(records),
        "type_counts": dict(type_counter),
        "errors": errors,
        "warnings": warnings,
        "is_valid": len(errors) == 0,
    }
    if errors:
        raise ValueError(json.dumps(report, ensure_ascii=False, indent=2)[:4000])
    return report


def load_variant_payload(variant_dir):
    variant_dir = Path(variant_dir)
    variant = variant_dir.name
    train = load_json_or_jsonl(variant_dir / "train.jsonl")
    valid_random = load_json_or_jsonl(variant_dir / "valid_random.jsonl")
    valid_grouped = load_json_or_jsonl(variant_dir / "valid_grouped.jsonl")
    reports = [
        validate_records(train, variant, "train"),
        validate_records(valid_random, variant, "valid_random"),
        validate_records(valid_grouped, variant, "valid_grouped"),
    ]
    return {
        "variant": variant,
        "variant_dir": variant_dir,
        "train": train,
        "valid_random": valid_random,
        "valid_grouped": valid_grouped,
        "variant_config": load_json_optional(variant_dir / "variant_config.json", {}),
        "metrics_pretrain": load_json_optional(variant_dir / "metrics_pretrain.json", {}),
        "schema_reports": reports,
    }


def record_sample_key(rec, seed):
    parts = [seed, rec.get("id"), rec.get("raw_index"), rec.get("original_question_hash"), rec.get("query", "")[:80]]
    return stable_int("|".join(map(str, parts)))


def stratified_sample(records, sample_count, seed, group_field="type"):
    records = list(records)
    if sample_count is None or sample_count >= len(records):
        return records
    groups = {}
    for rec in records:
        groups.setdefault(str(rec.get(group_field, "unknown")), []).append(rec)
    total = len(records)
    allocations = {}
    residuals = []
    for group, items in groups.items():
        exact = sample_count * len(items) / total
        base = int(math.floor(exact))
        if base == 0 and items:
            base = 1
        allocations[group] = min(base, len(items))
        residuals.append((exact - math.floor(exact), group))
    current = sum(allocations.values())
    for _, group in sorted(residuals, reverse=True):
        if current >= sample_count:
            break
        if allocations[group] < len(groups[group]):
            allocations[group] += 1
            current += 1
    while current > sample_count:
        removable = sorted((allocations[g], g) for g in allocations if allocations[g] > 0)
        count, group = removable[-1]
        allocations[group] -= 1
        current -= 1
    selected = []
    for group, items in groups.items():
        ordered = sorted(items, key=lambda rec: record_sample_key(rec, seed))
        selected.extend(ordered[: allocations[group]])
    selected = sorted(selected, key=lambda rec: record_sample_key(rec, seed))
    return selected[:sample_count]


def select_records_for_mode(payload, mode_cfg):
    train = stratified_sample(payload["train"], mode_cfg["train_sample_count"], SEED, "type")
    valid_random = stratified_sample(payload["valid_random"], mode_cfg["valid_random_sample_count"], SEED, "type")
    valid_grouped = stratified_sample(payload["valid_grouped"], mode_cfg["valid_grouped_sample_count"], SEED, "type")
    return {"train": train, "valid_random": valid_random, "valid_grouped": valid_grouped}


schema_summary = []
for variant_dir in VARIANT_DIRS:
    payload = load_variant_payload(variant_dir)
    for report in payload["schema_reports"]:
        schema_summary.append({
            "variant": report["variant"],
            "split": report["split"],
            "num_records": report["num_records"],
            "is_valid": report["is_valid"],
        })

schema_summary_df = pd.DataFrame(schema_summary)
display(schema_summary_df)

,variant,split,num_records,is_valid
0,v04_decimal_normalized,train,88908,True
1,v04_decimal_normalized,valid_random,4946,True
2,v04_decimal_normalized,valid_grouped,4964,True
3,v05_strip_asy,train,88893,True
4,v05_strip_asy,valid_random,4946,True
5,v05_strip_asy,valid_grouped,4964,True
6,v06_keep_asy_if_short,train,88907,True
7,v06_keep_asy_if_short,valid_random,4946,True
8,v06_keep_asy_if_short,valid_grouped,4964,True


In [6]:
ANCHOR_RE = re.compile(r"(?:Đáp án là|Câu trả lời là|The answer is|Answer|####)\s*[:：]",flags=re.IGNORECASE)
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")
SAFE_EVAL_NAMES = {"sqrt": math.sqrt, "pi": math.pi}


def clean_answer_tail(text):
    if text is None:
        return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    text = text.strip(" .。;；,，")
    return text or None


def extract_answer_text(text, allow_last_number=False):
    text = str(text or "")
    matches = list(ANCHOR_RE.finditer(text))
    if matches:
        return clean_answer_tail(text[matches[-1].end():])
    boxes = BOXED_RE.findall(text)
    if boxes:
        return clean_answer_tail(boxes[-1])
    if allow_last_number:
        nums = NUM_RE.findall(text)
        if nums:
            return clean_answer_tail(nums[-1])
    return None


def normalize_answer_text(answer):
    if answer is None:
        return ""
    text = str(answer).strip().lower()
    text = re.sub(r"\\boxed\{([^{}]+)\}", r"\1", text)
    text = text.replace("$", "")
    text = text.replace("\\,", "").replace("\\!", "")
    text = text.strip(" .。;；,，")
    text = re.sub(r"\s+", "", text)
    if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", text):
        text = text.replace(",", "")
    elif re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", text):
        text = text.replace(".", "").replace(",", ".")
    elif "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    return text


def parse_plain_number(text):
    text = str(text or "").strip().replace(" ", "")
    if not text:
        return None
    if "/" in text and re.fullmatch(r"[-+]?\d[\d.,]*/[-+]?\d[\d.,]*", text):
        left, right = text.split("/", 1)
        a = parse_plain_number(left)
        b = parse_plain_number(right)
        if a is not None and b not in (None, 0):
            return a / b
        return None
    if re.fullmatch(r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", text):
        text = text.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", text):
        text = text.replace(",", "")
    elif "," in text and "." not in text:
        right = text.split(",")[-1]
        text = text.replace(",", "") if len(right) == 3 else text.replace(",", ".")
    elif "," in text and "." in text:
        text = text.replace(",", "")
    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None


def looks_like_non_scalar_answer(answer):
    s = str(answer or "").strip()

    # tọa độ / tuple / list nhiều phần tử
    if re.fullmatch(r"\(?\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?(?:\s*,\s*-?\d+(?:\.\d+)?)*\s*\)?", s):
        return True

    # ma trận / vector
    if "pmatrix" in s or "bmatrix" in s or "matrix" in s:
        return True

    return False


def parse_number(answer):
    if answer is None:
        return None
    if looks_like_non_scalar_answer(answer):
        return None
    text = str(answer).strip()
    direct = parse_plain_number(text)
    if direct is not None:
        return direct
    text = re.sub(r"\\boxed\{([^{}]+)\}", r"\1", text)
    text = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", text)
    text = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", text)
    text = text.replace("\\pi", "pi")
    text = text.replace("^", "**")
    text = text.replace("\\cdot", "*").replace("\\times", "*")
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text)
    text = re.sub(r"(?<=\d),(?=\d)", ".", text)
    safe_text = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\s|e|E", "", text)
    if not safe_text:
        try:
            val = eval(text, {"__builtins__": {}}, SAFE_EVAL_NAMES)
            if isinstance(val, (int, float)) and math.isfinite(float(val)):
                return float(val)
        except Exception:
            pass
    m = NUM_RE.search(str(answer))
    return parse_plain_number(m.group(0)) if m else None


def relative_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))


def has_anchor(text):
    return bool(ANCHOR_RE.search(str(text or "")))


def anchor_last_line(text):
    lines = [line.strip() for line in str(text or "").splitlines() if line.strip()]
    return bool(lines and has_anchor(lines[-1]))


def repetition_issue(text):
    tokens = re.findall(r"\S+", str(text or "").lower())
    if len(tokens) < 24:
        return False
    grams = [tuple(tokens[i:i + 4]) for i in range(len(tokens) - 3)]
    counts = Counter(grams)
    return any(v >= 4 for v in counts.values())


def length_bucket(total_chars):
    if total_chars <= 250:
        return "short"
    if total_chars <= 700:
        return "medium"
    if total_chars <= 1400:
        return "long"
    return "very_long"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, local_files_only=LOCAL_FILES_ONLY)
if tokenizer.eos_token_id is None:
    tokenizer.eos_token = tokenizer.eos_token or "<|endoftext|>"
EOS_ID = int(tokenizer.eos_token_id) if tokenizer.eos_token_id is not None else 50256
PAD_ID = EOS_ID
tokenizer.pad_token_id = PAD_ID
if getattr(tokenizer, "pad_token", None) is None and getattr(tokenizer, "eos_token", None) is not None:
    tokenizer.pad_token = tokenizer.eos_token


def build_prompt(rec):
    return INFERENCE_PREFIX.format(query=str(rec.get("query", "")).strip())


def encode_no_special(text):
    return tokenizer(str(text or ""), add_special_tokens=False)["input_ids"]


def fit_prompt_response(prompt_ids, response_ids, max_length):
    if len(prompt_ids) + len(response_ids) <= max_length:
        return prompt_ids, response_ids
    response_room = min(len(response_ids), max(1, max_length // 2))
    prompt_room = max_length - response_room
    if prompt_room <= 0:
        response_room = max_length
        prompt_room = 0
    prompt_ids = prompt_ids[:prompt_room]
    response_ids = response_ids[-response_room:]
    return prompt_ids, response_ids


class MathSFTDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.records = list(records)
        self.tokenizer = tokenizer
        self.max_length = int(max_length)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        prompt_ids = encode_no_special(build_prompt(rec))
        response_ids = encode_no_special(str(rec.get("response", "")).strip()) + [EOS_ID]
        prompt_ids, response_ids = fit_prompt_response(prompt_ids, response_ids, self.max_length)
        input_ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }


@dataclass
class PadCollator:
    pad_id: int
    pad_to_multiple_of: int = 8

    def __call__(self, batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        if self.pad_to_multiple_of:
            max_len = int(math.ceil(max_len / self.pad_to_multiple_of) * self.pad_to_multiple_of)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for item in batch:
            pad = max_len - len(item["input_ids"])
            out["input_ids"].append(item["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(item["attention_mask"] + [0] * pad)
            out["labels"].append(item["labels"] + [-100] * pad)
        return {key: torch.tensor(value, dtype=torch.long) for key, value in out.items()}


def label_masking_debug_samples(records, max_length, count=5):
    dataset = MathSFTDataset(records[:count], tokenizer, max_length)
    samples = []
    for idx in range(min(count, len(dataset))):
        rec = records[idx]
        item = dataset[idx]
        raw_prompt_ids = encode_no_special(build_prompt(rec))
        raw_response_ids = encode_no_special(str(rec.get("response", "")).strip()) + [EOS_ID]
        fitted_prompt_ids, fitted_response_ids = fit_prompt_response(raw_prompt_ids, raw_response_ids, max_length)
        labels = item["labels"]
        prompt_token_count = len(fitted_prompt_ids)
        samples.append({
            "id": rec.get("id"),
            "prompt_text": build_prompt(rec),
            "response_text": str(rec.get("response", ""))[:500],
            "prompt_token_count": prompt_token_count,
            "total_token_count": len(item["input_ids"]),
            "num_label_tokens": int(sum(1 for label in labels if label != -100)),
            "masking_is_valid": bool(labels and labels[:prompt_token_count] == [-100] * prompt_token_count and labels[prompt_token_count:] == fitted_response_ids),
        })
    return samples


print("Tokenizer length:", len(tokenizer), "| EOS_ID:", EOS_ID, "| PAD_ID:", PAD_ID)

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer length: 50258 | EOS_ID: 50257 | PAD_ID: 50257


In [8]:
def ensure_model_embeddings(model):
    if len(tokenizer) > model.get_input_embeddings().num_embeddings:
        model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = PAD_ID
    model.config.eos_token_id = EOS_ID
    return model


def make_training_args(run_dir, mode_cfg):
    kwargs = {
        "output_dir": str(run_dir / "trainer_tmp"),
        "num_train_epochs": mode_cfg["epochs"],
        "per_device_train_batch_size": mode_cfg["per_device_train_batch_size"],
        "per_device_eval_batch_size": mode_cfg["per_device_eval_batch_size"],
        "gradient_accumulation_steps": mode_cfg["gradient_accumulation_steps"],
        "learning_rate": mode_cfg["learning_rate"],
        "warmup_ratio": mode_cfg["warmup_ratio"],
        "lr_scheduler_type": "cosine",
        "weight_decay": mode_cfg["weight_decay"],
        "logging_steps": mode_cfg["logging_steps"],
        "save_strategy": "no",
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "remove_unused_columns": False,
        "dataloader_num_workers": 2 if IS_KAGGLE else 0,
        "gradient_checkpointing": True,
        "max_grad_norm": 1.0,
    }
    if mode_cfg["max_steps"] is not None:
        kwargs["max_steps"] = int(mode_cfg["max_steps"])
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "no"
    else:
        kwargs["evaluation_strategy"] = "no"
    if "bf16" in sig.parameters:
        kwargs["bf16"] = use_bf16
    if "fp16" in sig.parameters:
        kwargs["fp16"] = use_fp16
    if not torch.cuda.is_available() and "use_cpu" in sig.parameters:
        kwargs["use_cpu"] = True
    return TrainingArguments(**kwargs)


def build_lora_config(mode_cfg):
    return LoraConfig(
        task_type="CAUSAL_LM",
        r=int(mode_cfg["lora_r"]),
        lora_alpha=int(mode_cfg["lora_alpha"]),
        lora_dropout=float(mode_cfg["lora_dropout"]),
        target_modules=LORA_TARGET_MODULES,
        fan_in_fan_out=True,
    )


def load_lora_model(mode_cfg):
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME_OR_PATH, local_files_only=LOCAL_FILES_ONLY)
    model = ensure_model_embeddings(model)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model = get_peft_model(model, build_lora_config(mode_cfg))
    return model


def build_run_id(variant):
    return f"{RUN_MODE_PREFIX[RUN_MODE]}_{variant}_seed{SEED}"


def build_run_config(payload, selected_records, mode_cfg):
    variant = payload["variant"]
    run_id = build_run_id(variant)
    return {
        "run_id": run_id,
        "variant": variant,
        "run_mode": RUN_MODE,
        "seed": SEED,
        "data": {
            "train_path": str(payload["variant_dir"] / "train.jsonl"),
            "valid_random_path": str(payload["variant_dir"] / "valid_random.jsonl"),
            "valid_grouped_path": str(payload["variant_dir"] / "valid_grouped.jsonl"),
            "train_sample_count": len(selected_records["train"]),
            "valid_random_sample_count": len(selected_records["valid_random"]),
            "valid_grouped_sample_count": len(selected_records["valid_grouped"]),
            "sampling_strategy": SAMPLING_STRATEGY,
        },
        "model": {
            "base_model_name": BASE_MODEL_NAME,
            "model_load_path": MODEL_NAME_OR_PATH,
            "model_class": "AutoModelForCausalLM",
            "tokenizer_name": MODEL_NAME_OR_PATH,
            "local_files_only": LOCAL_FILES_ONLY,
        },
        "prompt": {
            "template_name": TEMPLATE_NAME,
            "template": PROMPT_TEMPLATE,
            "inference_prefix": INFERENCE_PREFIX,
            "mask_prompt_loss": MASK_PROMPT_LOSS,
        },
        "tokenization": {
            "max_length": mode_cfg["max_length"],
            "truncation_policy": "preserve_prompt_prefix_and_response_tail",
            "pad_token_policy": "eos_as_pad",
            "dynamic_padding": True,
        },
        "training": {
            "method": "lora",
            "epochs": mode_cfg["epochs"],
            "max_steps": mode_cfg["max_steps"],
            "learning_rate": mode_cfg["learning_rate"],
            "batch_size": mode_cfg["per_device_train_batch_size"],
            "gradient_accumulation_steps": mode_cfg["gradient_accumulation_steps"],
            "warmup_ratio": mode_cfg["warmup_ratio"],
            "weight_decay": mode_cfg["weight_decay"],
            "fp16_or_bf16": MIXED_PRECISION,
        },
        "lora": {
            "enabled": True,
            "task_type": "CAUSAL_LM",
            "r": mode_cfg["lora_r"],
            "alpha": mode_cfg["lora_alpha"],
            "dropout": mode_cfg["lora_dropout"],
            "target_modules": LORA_TARGET_MODULES,
        },
        "generation": {
            "max_new_tokens": mode_cfg["max_new_tokens"],
            "do_sample": GENERATION_CONFIG["do_sample"],
            "num_beams": GENERATION_CONFIG["num_beams"],
            "temperature": GENERATION_CONFIG["temperature"],
            "top_p": GENERATION_CONFIG["top_p"],
            "eos_token_id": EOS_ID,
            "pad_token_id": PAD_ID,
        },
        "variant_config": payload["variant_config"],
        "metrics_pretrain": payload["metrics_pretrain"],
    }


def eval_loss_on_records(trainer, records, mode_cfg):
    if not records:
        return {"eval_loss": None, "perplexity": None}
    count = mode_cfg.get("trainer_eval_sample_count")
    eval_records = records[:count] if count else records
    eval_ds = MathSFTDataset(eval_records, tokenizer, mode_cfg["max_length"])
    metrics = trainer.evaluate(eval_dataset=eval_ds, metric_key_prefix="eval")
    loss = metrics.get("eval_loss")
    if loss is None:
        return {"eval_loss": None, "perplexity": None}
    perplexity = math.exp(loss) if loss < 20 else float("inf")
    return {"eval_loss": float(loss), "perplexity": float(perplexity)}

In [9]:
def postprocess_generated_tail(text):
    text = str(text or "").strip()
    for marker in ["\nCâu hỏi:", "\nQuestion:", "\n###"]:
        pos = text.find(marker)
        if pos >= 0:
            text = text[:pos].strip()
    return text


def make_prediction_record(run_config, rec, split, prompt, generated_tail, generated_token_count, reached_max_new_tokens):
    gold_answer = rec.get("final_answer")
    pred_answer = extract_answer_text(generated_tail, allow_last_number=False)
    pred_norm = normalize_answer_text(pred_answer)
    gold_norm = normalize_answer_text(gold_answer)
    pred_num = parse_number(pred_answer)
    gold_num = parse_number(gold_answer)
    rel = relative_error(pred_num, gold_num)
    query_len = len(str(rec.get("query", "")))
    response_len = len(str(rec.get("response", "")))
    generated_len = len(str(generated_tail))
    return {
        "run_id": run_config["run_id"],
        "variant": run_config["variant"],
        "split": split,
        "id": rec.get("id"),
        "raw_index": rec.get("raw_index"),
        "type": rec.get("type"),
        "source_group": rec.get("source_group"),
        "aug_type": rec.get("aug_type"),
        "query": rec.get("query"),
        "gold_response": rec.get("response"),
        "gold_final_answer": gold_answer,
        "prompt": prompt,
        "generated_text": prompt + generated_tail,
        "generated_tail": generated_tail,
        "pred_final_answer_raw": pred_answer,
        "pred_final_answer_normalized": pred_norm,
        "gold_final_answer_normalized": gold_norm,
        "answer_extracted": pred_answer is not None,
        "format_has_anchor": has_anchor(generated_tail),
        "format_anchor_last_line": anchor_last_line(generated_tail),
        "is_exact_match": bool(pred_norm and gold_norm and pred_norm == gold_norm),
        "is_numeric_comparable": pred_num is not None and gold_num is not None,
        "relative_error": rel,
        "pass_rel_error_1e_3": bool(rel is not None and rel <= 1e-3),
        "pass_rel_error_1e_2": bool(rel is not None and rel <= 1e-2),
        "query_char_len": query_len,
        "gold_response_char_len": response_len,
        "generated_char_len": generated_len,
        "generated_token_count": int(generated_token_count),
        "truncated_generation": bool(reached_max_new_tokens),
        "repetition_issue": repetition_issue(generated_tail),
        "length_bucket": length_bucket(query_len + response_len),
        "has_latex_original": bool(rec.get("has_latex_original", False)),
        "has_latex_after": bool(rec.get("has_latex_after", False)),
        "has_asy_original": bool(rec.get("has_asy_original", False)),
        "has_asy_after": bool(rec.get("has_asy_after", False)),
        "was_asy_stripped": bool(rec.get("was_asy_stripped", False)),
        "was_decimal_normalized": bool(rec.get("was_decimal_normalized", False)),
        "was_anchor_rebuilt": bool(rec.get("was_anchor_rebuilt", False)),
        "answer_extract_method": rec.get("answer_extract_method"),
        "long_query": bool(query_len > 300),
        "long_response": bool(response_len > 800),
    }


def generate_predictions(model, records, run_config, split, mode_cfg):
    device = next(model.parameters()).device
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    predictions = []
    batch_size = int(mode_cfg["generation_batch_size"])
    max_new_tokens = int(mode_cfg["max_new_tokens"])
    model.eval()
    with torch.inference_mode():
        for start in tqdm(range(0, len(records), batch_size), desc=f"generate {split}"):
            batch = records[start:start + batch_size]
            prompts = [build_prompt(rec) for rec in batch]
            enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=mode_cfg["max_length"])
            enc = {k: v.to(device) for k, v in enc.items()}
            input_width = enc["input_ids"].shape[1]
            gen_kwargs = {
                "max_new_tokens": max_new_tokens,
                "do_sample": GENERATION_CONFIG["do_sample"],
                "num_beams": GENERATION_CONFIG["num_beams"],
                "repetition_penalty": GENERATION_CONFIG["repetition_penalty"],
                "no_repeat_ngram_size": GENERATION_CONFIG["no_repeat_ngram_size"],
                "pad_token_id": PAD_ID,
                "eos_token_id": EOS_ID,
            }
            generated = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc.get("attention_mask"),
                **gen_kwargs,
            )
            for row_idx, rec in enumerate(batch):
                new_tokens = generated[row_idx, input_width:]
                token_list = new_tokens.detach().cpu().tolist()
                if EOS_ID in token_list:
                    token_count = token_list.index(EOS_ID) + 1
                else:
                    token_count = len(token_list)
                tail = tokenizer.decode(new_tokens, skip_special_tokens=True)
                tail = postprocess_generated_tail(tail)
                reached_max = len(token_list) >= max_new_tokens and EOS_ID not in token_list
                predictions.append(make_prediction_record(run_config, rec, split, prompts[row_idx], tail, token_count, reached_max))
    tokenizer.padding_side = old_padding_side
    return predictions

In [10]:
def rate(values):
    values = list(values)
    return float(np.mean(values)) if values else 0.0


def nullable_mean(values):
    values = [x for x in values if x is not None and not pd.isna(x)]
    return float(np.mean(values)) if values else None


def nullable_median(values):
    values = [x for x in values if x is not None and not pd.isna(x)]
    return float(np.median(values)) if values else None


def compute_split_metrics(predictions, run_config, split, loss_metrics):
    n = len(predictions)
    rel_values = [p.get("relative_error") for p in predictions if p.get("relative_error") is not None]
    return {
        "run_id": run_config["run_id"],
        "variant": run_config["variant"],
        "split": split,
        "num_samples": n,
        "loss_metrics": loss_metrics,
        "format_metrics": {
            "format_has_anchor_rate": rate(p.get("format_has_anchor") for p in predictions),
            "anchor_last_line_rate": rate(p.get("format_anchor_last_line") for p in predictions),
            "answer_extraction_rate": rate(p.get("answer_extracted") for p in predictions),
            "empty_generation_rate": rate(not str(p.get("generated_tail", "")).strip() for p in predictions),
            "truncated_generation_rate": rate(p.get("truncated_generation") for p in predictions),
        },
        "answer_metrics": {
            "exact_match_normalized": rate(p.get("is_exact_match") for p in predictions),
            "numeric_comparable_rate": rate(p.get("is_numeric_comparable") for p in predictions),
            "relative_error_mean": nullable_mean(rel_values),
            "relative_error_median": nullable_median(rel_values),
            "pass_rel_error_1e_3": rate(p.get("pass_rel_error_1e_3") for p in predictions),
            "pass_rel_error_1e_2": rate(p.get("pass_rel_error_1e_2") for p in predictions),
        },
        "generation_metrics": {
            "avg_generated_chars": nullable_mean([p.get("generated_char_len") for p in predictions]),
            "avg_generated_tokens": nullable_mean([p.get("generated_token_count") for p in predictions]),
            "repetition_issue_rate": rate(p.get("repetition_issue") for p in predictions),
        },
    }


def compact_prediction_example(pred):
    return {
        "id": pred.get("id"),
        "type": pred.get("type"),
        "query": str(pred.get("query", ""))[:500],
        "gold_answer": pred.get("gold_final_answer"),
        "pred_answer": pred.get("pred_final_answer_raw"),
        "relative_error": pred.get("relative_error"),
        "generated_tail": str(pred.get("generated_tail", ""))[-700:],
    }


def build_error_analysis(predictions):
    groups = {
        "no_anchor_generated": lambda p: not p.get("format_has_anchor"),
        "answer_not_extracted": lambda p: not p.get("answer_extracted"),
        "numeric_parse_failed": lambda p: p.get("answer_extracted") and not p.get("is_numeric_comparable"),
        "high_relative_error": lambda p: p.get("relative_error") is not None and p.get("relative_error") > 0.1,
        "exact_mismatch_but_numeric_close": lambda p: not p.get("is_exact_match") and p.get("pass_rel_error_1e_2"),
        "generated_too_short": lambda p: p.get("generated_char_len", 0) < 20,
        "generated_too_long": lambda p: p.get("generated_char_len", 0) > 1500 or p.get("truncated_generation"),
        "repeated_text": lambda p: p.get("repetition_issue"),
        "latex_answer_failed": lambda p: p.get("has_latex_original") and not p.get("pass_rel_error_1e_3"),
        "decimal_answer_failed": lambda p: p.get("was_decimal_normalized") and not p.get("pass_rel_error_1e_3"),
    }
    out = {}
    n = len(predictions)
    for name, fn in groups.items():
        items = [p for p in predictions if fn(p)]
        out[name] = {
            "count": len(items),
            "rate": len(items) / n if n else 0.0,
            "examples": [compact_prediction_example(p) for p in items[:20]],
        }
    return out


def render_sample_item(pred):
    rel = pred.get("relative_error")
    rel_text = "None" if rel is None else f"{rel:.6g}"
    parts = [
        f"- id: {pred.get('id')}",
        f"  type: {pred.get('type')}",
        f"  gold answer: {pred.get('gold_final_answer')}",
        f"  pred answer: {pred.get('pred_final_answer_raw')}",
        f"  relative error: {rel_text}",
        f"  query: {str(pred.get('query', ''))[:350]}",
        f"  generated tail: {str(pred.get('generated_tail', ''))[-500:]}",
    ]
    return "\n".join(parts)


def write_sample_generations(predictions_by_split, path):
    all_preds = []
    for split_preds in predictions_by_split.values():
        all_preds.extend(split_preds)
    sections = [
        ("Correct Examples", [p for p in all_preds if p.get("pass_rel_error_1e_3")][:10]),
        ("Wrong Examples", [p for p in all_preds if not p.get("pass_rel_error_1e_3")][:20]),
        ("No Anchor Examples", [p for p in all_preds if not p.get("format_has_anchor")][:10]),
        ("MATH / LaTeX Examples", [p for p in all_preds if str(p.get("type", "")).startswith("MATH") or p.get("has_latex_original")][:10]),
        ("FOBAR / SV Examples", [p for p in all_preds if "FOBAR" in str(p.get("type", "")) or "SV" in str(p.get("type", ""))][:10]),
        ("Long Examples", sorted(all_preds, key=lambda p: p.get("query_char_len", 0) + p.get("gold_response_char_len", 0), reverse=True)[:10]),
    ]
    lines = ["# Sample Generations", ""]
    for title, items in sections:
        lines.extend([f"## {title}", ""])
        if items:
            lines.extend(render_sample_item(item) for item in items)
        else:
            lines.append("Không có mẫu phù hợp.")
        lines.append("")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines), encoding="utf-8")

In [11]:
def run_is_complete(run_dir):
    run_dir = Path(run_dir)
    required = [
        run_dir / "run_config.json",
        run_dir / "eval_valid_random_metrics.json",
        run_dir / "eval_valid_grouped_metrics.json",
        run_dir / "predictions_valid_random.jsonl",
        run_dir / "predictions_valid_grouped.jsonl",
    ]
    return all(path.exists() for path in required)


def train_and_evaluate_variant(variant_dir):
    mode_cfg = MODE_CONFIGS[RUN_MODE]
    payload = load_variant_payload(variant_dir)
    selected = select_records_for_mode(payload, mode_cfg)
    run_config = build_run_config(payload, selected, mode_cfg)
    run_id = run_config["run_id"]
    run_dir = OUTPUT_ROOT / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    if run_is_complete(run_dir) and not FORCE_RERUN:
        print("Skip existing run:", run_id)
        return {"run_id": run_id, "variant": payload["variant"], "run_dir": str(run_dir), "skipped": True}
    save_json(run_config, run_dir / "run_config.json")
    save_json(label_masking_debug_samples(selected["train"], mode_cfg["max_length"]), run_dir / "label_masking_debug_samples.json")
    if not torch.cuda.is_available() and not ALLOW_CPU_TRAINING:
        raise RuntimeError("Không có CUDA. Nếu chỉ test pipeline rất nhỏ trên CPU, đặt ALLOW_CPU_TRAINING=True.")
    collator = PadCollator(PAD_ID)
    train_ds = MathSFTDataset(selected["train"], tokenizer, mode_cfg["max_length"])
    model = load_lora_model(mode_cfg)
    training_args = make_training_args(run_dir, mode_cfg)
    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, data_collator=collator)
    start = time.time()
    train_output = trainer.train()
    train_minutes = (time.time() - start) / 60
    save_jsonl(trainer.state.log_history, run_dir / "train_log.jsonl")
    trainer.state.save_to_json(str(run_dir / "trainer_state.json"))
    adapter_dir = run_dir / "adapter"
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    loss_random = eval_loss_on_records(trainer, selected["valid_random"], mode_cfg)
    loss_grouped = eval_loss_on_records(trainer, selected["valid_grouped"], mode_cfg)
    predictions_by_split = {}
    metrics_by_split = {}
    for split, records, loss_metrics in [
        ("valid_random", selected["valid_random"], loss_random),
        ("valid_grouped", selected["valid_grouped"], loss_grouped),
    ]:
        preds = generate_predictions(trainer.model, records, run_config, split, mode_cfg)
        predictions_by_split[split] = preds
        metrics = compute_split_metrics(preds, run_config, split, loss_metrics)
        metrics_by_split[split] = metrics
        save_jsonl(preds, run_dir / f"predictions_{split}.jsonl")
        save_json(metrics, run_dir / f"eval_{split}_metrics.json")
        save_json(build_error_analysis(preds), run_dir / f"error_analysis_{split}.json")
    write_sample_generations(predictions_by_split, run_dir / "sample_generations.md")
    result = {
        "run_id": run_id,
        "variant": payload["variant"],
        "run_dir": str(run_dir),
        "skipped": False,
        "train_runtime_minutes": train_minutes,
        "train_output": train_output.metrics,
        "metrics": metrics_by_split,
    }
    if RUN_MODE == "sanity_check":
        save_json(result, OUTPUT_ROOT / "sanity_check_report.json")
        sample_md = run_dir / "sample_generations.md"
        if sample_md.exists():
            (OUTPUT_ROOT / "sanity_sample_generations.md").write_text(sample_md.read_text(encoding="utf-8"), encoding="utf-8")
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return result


run_results = []
for variant_dir in VARIANT_DIRS:
    print("=" * 100)
    print("Run variant:", variant_dir.name)
    run_results.append(train_and_evaluate_variant(variant_dir))

save_json(run_results, OUTPUT_ROOT / "run_results_raw.json")
run_results

Run variant: v04_decimal_normalized


pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: NlpHUST/gpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


...

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


generate valid_random:   0%|          | 0/250 [00:00<?, ?it/s]

<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<string>:1: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?


generate valid_grouped:   0%|          | 0/250 [00:00<?, ?it/s]

[{'run_id': 'light_v04_decimal_normalized_seed42',
  'variant': 'v04_decimal_normalized',
  'run_dir': '/kaggle/working/variant_experiments/runs/light_v04_decimal_normalized_seed42',
  'skipped': False,
  'train_runtime_minutes': 12.237598983446757,
  'train_output': {'train_runtime': 733.929,
   'train_samples_per_second': 13.625,
   'train_steps_per_second': 0.426,
   'total_flos': 1927530464477184.0,
   'train_loss': 2.0188748783196884,
   'epoch': 1.0},
  'metrics': {'valid_random': {'run_id': 'light_v04_decimal_normalized_seed42',
    'variant': 'v04_decimal_normalized',
    'split': 'valid_random',
    'num_samples': 1000,
    'loss_metrics': {'eval_loss': 1.890362024307251,
     'perplexity': 6.621765487229143},
    'format_metrics': {'format_has_anchor_rate': 0.711,
     'anchor_last_line_rate': 0.45,
     'answer_extraction_rate': 0.755,
     'empty_generation_rate': 0.0,
     'truncated_generation_rate': 1.0},
    'answer_metrics': {'exact_match_normalized': 0.002,
     'nume

In [12]:
def load_run_metrics(run_dir, split):
    return load_json_optional(Path(run_dir) / f"eval_{split}_metrics.json", {})


def flatten_run_summary(run_dir):
    run_config = load_json_optional(Path(run_dir) / "run_config.json", {})
    row = {
        "run_id": run_config.get("run_id", Path(run_dir).name),
        "variant": run_config.get("variant"),
        "run_mode": run_config.get("run_mode"),
        "seed": run_config.get("seed"),
    }
    for split in ["valid_random", "valid_grouped"]:
        metrics = load_run_metrics(run_dir, split)
        row[f"{split}_num_samples"] = metrics.get("num_samples")
        row[f"{split}_eval_loss"] = metrics.get("loss_metrics", {}).get("eval_loss")
        row[f"{split}_perplexity"] = metrics.get("loss_metrics", {}).get("perplexity")
        row[f"{split}_format_has_anchor_rate"] = metrics.get("format_metrics", {}).get("format_has_anchor_rate")
        row[f"{split}_answer_extraction_rate"] = metrics.get("format_metrics", {}).get("answer_extraction_rate")
        row[f"{split}_exact_match_normalized"] = metrics.get("answer_metrics", {}).get("exact_match_normalized")
        row[f"{split}_numeric_comparable_rate"] = metrics.get("answer_metrics", {}).get("numeric_comparable_rate")
        row[f"{split}_pass_rel_error_1e_3"] = metrics.get("answer_metrics", {}).get("pass_rel_error_1e_3")
        row[f"{split}_pass_rel_error_1e_2"] = metrics.get("answer_metrics", {}).get("pass_rel_error_1e_2")
        row[f"{split}_relative_error_mean"] = metrics.get("answer_metrics", {}).get("relative_error_mean")
        row[f"{split}_repetition_issue_rate"] = metrics.get("generation_metrics", {}).get("repetition_issue_rate")
    return row


def load_predictions_file(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def group_metric_rows(predictions, group_col):
    if not predictions:
        return []
    df = pd.DataFrame(predictions)
    rows = []
    for keys, part in df.groupby(["run_id", "variant", "split", group_col], dropna=False):
        run_id, variant, split, group_value = keys
        rows.append({
            "run_id": run_id,
            "variant": variant,
            "split": split,
            group_col: group_value,
            "num_samples": len(part),
            "answer_extraction_rate": float(part["answer_extracted"].mean()),
            "exact_match_normalized": float(part["is_exact_match"].mean()),
            "numeric_comparable_rate": float(part["is_numeric_comparable"].mean()) if "is_numeric_comparable" in part else None,
            "pass_rel_error_1e_3": float(part["pass_rel_error_1e_3"].mean()),
            "pass_rel_error_1e_2": float(part["pass_rel_error_1e_2"].mean()),
            "avg_relative_error": nullable_mean(part["relative_error"].tolist()) if "relative_error" in part else None,
        })
    return rows


def feature_subset_rows(predictions):
    features = ["has_latex_original", "has_asy_original", "was_decimal_normalized", "long_query", "long_response"]
    rows = []
    if not predictions:
        return rows
    df = pd.DataFrame(predictions)
    for feature in features:
        for keys, part in df.groupby(["run_id", "variant", "split", feature], dropna=False):
            run_id, variant, split, value = keys
            rows.append({
                "run_id": run_id,
                "variant": variant,
                "split": split,
                "feature_name": feature,
                "feature_value": bool(value),
                "num_samples": len(part),
                "answer_extraction_rate": float(part["answer_extracted"].mean()),
                "exact_match_normalized": float(part["is_exact_match"].mean()),
                "pass_rel_error_1e_3": float(part["pass_rel_error_1e_3"].mean()),
                "pass_rel_error_1e_2": float(part["pass_rel_error_1e_2"].mean()),
            })
    return rows


def build_ranking(summary_df):
    if summary_df.empty:
        return pd.DataFrame()
    df = summary_df.copy()
    df["gap_random_minus_grouped"] = df["valid_random_pass_rel_error_1e_3"].fillna(0) - df["valid_grouped_pass_rel_error_1e_3"].fillna(0)
    df = df.sort_values(
        by=[
            "valid_grouped_pass_rel_error_1e_3",
            "valid_grouped_exact_match_normalized",
            "valid_grouped_answer_extraction_rate",
            "valid_grouped_format_has_anchor_rate",
            "gap_random_minus_grouped",
        ],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
    return pd.DataFrame({
        "rank": list(range(1, len(df) + 1)),
        "variant": df["variant"],
        "run_id": df["run_id"],
        "valid_grouped_pass_rel_error_1e_3": df["valid_grouped_pass_rel_error_1e_3"],
        "valid_grouped_exact_match": df["valid_grouped_exact_match_normalized"],
        "valid_grouped_answer_extraction_rate": df["valid_grouped_answer_extraction_rate"],
        "valid_grouped_format_anchor_rate": df["valid_grouped_format_has_anchor_rate"],
        "valid_random_pass_rel_error_1e_3": df["valid_random_pass_rel_error_1e_3"],
        "gap_random_minus_grouped": df["gap_random_minus_grouped"],
        "notes": ["ranked_by_valid_grouped" for _ in range(len(df))],
    })


def get_nested(obj, path):
    cur = obj
    for part in path.split("."):
        if not isinstance(cur, dict):
            return None
        cur = cur.get(part)
    return cur


def build_config_consistency_report(run_dirs):
    invariant_paths = [
        "run_mode",
        "seed",
        "data.train_sample_count",
        "data.valid_random_sample_count",
        "data.valid_grouped_sample_count",
        "data.sampling_strategy",
        "model.base_model_name",
        "model.model_load_path",
        "model.model_class",
        "model.tokenizer_name",
        "prompt.template_name",
        "prompt.template",
        "prompt.inference_prefix",
        "prompt.mask_prompt_loss",
        "tokenization.max_length",
        "tokenization.truncation_policy",
        "tokenization.pad_token_policy",
        "tokenization.dynamic_padding",
        "training.method",
        "training.epochs",
        "training.max_steps",
        "training.learning_rate",
        "training.batch_size",
        "training.gradient_accumulation_steps",
        "training.warmup_ratio",
        "training.weight_decay",
        "training.fp16_or_bf16",
        "lora.enabled",
        "lora.task_type",
        "lora.r",
        "lora.alpha",
        "lora.dropout",
        "lora.target_modules",
        "generation.max_new_tokens",
        "generation.do_sample",
        "generation.num_beams",
        "generation.temperature",
        "generation.top_p",
        "generation.eos_token_id",
        "generation.pad_token_id",
    ]
    configs = []
    for run_dir in run_dirs:
        cfg = load_json_optional(Path(run_dir) / "run_config.json", {})
        if cfg:
            configs.append(cfg)
    checks = []
    for path in invariant_paths:
        values = [get_nested(cfg, path) for cfg in configs]
        serialized = [json.dumps(v, ensure_ascii=False, sort_keys=True, default=json_default) for v in values]
        checks.append({
            "field": path,
            "is_consistent": len(set(serialized)) <= 1,
            "values_by_run": {cfg.get("run_id"): get_nested(cfg, path) for cfg in configs},
        })
    return {
        "run_mode": RUN_MODE,
        "num_runs_checked": len(configs),
        "fair_ablation_mode": RUN_MODE in {"light_ablation", "medium_ablation"},
        "all_invariant_fields_consistent": all(item["is_consistent"] for item in checks),
        "note": "strong_final_train có thể dùng config mạnh hơn và không dùng để kết luận preprocessing công bằng giữa variants.",
        "checks": checks,
    }


comparison_dir = OUTPUT_ROOT / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
run_dirs = sorted((OUTPUT_ROOT / "runs").glob(f"{RUN_MODE_PREFIX[RUN_MODE]}_*_seed{SEED}"))
summary_rows = [flatten_run_summary(run_dir) for run_dir in run_dirs if run_is_complete(run_dir)]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(comparison_dir / "all_runs_summary.csv", index=False, encoding="utf-8-sig")
save_json(summary_rows, comparison_dir / "all_runs_summary.json")

all_predictions = []
for run_dir in run_dirs:
    all_predictions.extend(load_predictions_file(run_dir / "predictions_valid_random.jsonl"))
    all_predictions.extend(load_predictions_file(run_dir / "predictions_valid_grouped.jsonl"))

pd.DataFrame(group_metric_rows(all_predictions, "type")).to_csv(comparison_dir / "metric_by_type.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(group_metric_rows(all_predictions, "source_group")).to_csv(comparison_dir / "metric_by_source_group.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(group_metric_rows(all_predictions, "aug_type")).to_csv(comparison_dir / "metric_by_aug_type.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(group_metric_rows(all_predictions, "length_bucket")).to_csv(comparison_dir / "metric_by_length_bucket.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(feature_subset_rows(all_predictions)).to_csv(comparison_dir / "metric_by_feature_subset.csv", index=False, encoding="utf-8-sig")

ranking_df = build_ranking(summary_df)
ranking_df.to_csv(comparison_dir / "variant_ranking_by_valid_grouped.csv", index=False, encoding="utf-8-sig")
config_consistency_report = build_config_consistency_report([run_dir for run_dir in run_dirs if run_is_complete(run_dir)])
save_json(config_consistency_report, comparison_dir / "config_consistency_report.json")

if not ranking_df.empty:
    best = ranking_df.iloc[0].to_dict()
    report_lines = [
        "# Best Variant Report",
        "",
        "## Selected Variant",
        str(best.get("variant")),
        "",
        "## Why this variant is selected",
        "Variant được chọn theo thứ tự ưu tiên valid_grouped pass_rel_error_1e_3, exact match, extraction rate, anchor rate và gap random-grouped nhỏ.",
        "",
        "## Key Metrics",
        json.dumps(best, ensure_ascii=False, indent=2),
        "",
        "## Weaknesses",
        "Xem error_analysis_valid_grouped.json và sample_generations.md trong run folder tương ứng.",
        "",
        "## Error Groups",
        "Các nhóm lỗi đã lưu: no_anchor_generated, answer_not_extracted, numeric_parse_failed, high_relative_error, exact_mismatch_but_numeric_close, generated_too_short, generated_too_long, repeated_text, latex_answer_failed, decimal_answer_failed.",
        "",
        "## Recommended next run",
        "Chạy medium_ablation cho top 2-3 variant nếu đây là light_ablation; chạy strong_final_train cho variant đứng đầu nếu medium_ablation đã ổn định.",
    ]
    recommended = {
        "next_run_mode": "medium_ablation" if RUN_MODE == "light_ablation" else "strong_final_train",
        "recommended_variants": ranking_df["variant"].head(3).tolist() if RUN_MODE == "light_ablation" else ranking_df["variant"].head(1).tolist(),
    }
else:
    report_lines = ["# Best Variant Report", "", "Chưa có run hoàn chỉnh để ranking."]
    recommended = {"next_run_mode": None, "recommended_variants": []}

(comparison_dir / "best_variant_report.md").write_text("\n".join(report_lines), encoding="utf-8")
save_json(recommended, comparison_dir / "recommended_next_runs.json")

display(summary_df)
display(ranking_df)

,run_id,variant,run_mode,seed,valid_random_num_samples,valid_random_eval_loss,valid_random_perplexity,valid_random_format_has_anchor_rate,valid_random_answer_extraction_rate,valid_random_exact_match_normalized,...,valid_grouped_eval_loss,valid_grouped_perplexity,valid_grouped_format_has_anchor_rate,valid_grouped_answer_extraction_rate,valid_grouped_exact_match_normalized,valid_grouped_numeric_comparable_rate,valid_grouped_pass_rel_error_1e_3,valid_grouped_pass_rel_error_1e_2,valid_grouped_relative_error_mean,valid_grouped_repetition_issue_rate
0,light_v04_decimal_normalized_seed42,v04_decimal_normalized,light_ablation,42,1000,1.890362,6.621765,0.711,0.755,0.002,...,1.843533,6.318826,0.685,0.731,0.001,0.642,0.019,0.019,6.997769,0.016
1,light_v05_strip_asy_seed42,v05_strip_asy,light_ablation,42,1000,1.889068,6.613202,0.726,0.764,0.004,...,1.840790,6.301516,0.716,0.756,0.003,0.678,0.015,0.015,144350.660808,0.017
2,light_v06_keep_asy_if_short_seed42,v06_keep_asy_if_short,light_ablation,42,1000,1.890761,6.624409,0.727,0.769,0.002,...,1.842217,6.310515,0.717,0.757,0.004,0.681,0.017,0.017,4.550385,0.015


,rank,variant,run_id,valid_grouped_pass_rel_error_1e_3,valid_grouped_exact_match,valid_grouped_answer_extraction_rate,valid_grouped_format_anchor_rate,valid_random_pass_rel_error_1e_3,gap_random_minus_grouped,notes
0,1,v04_decimal_normalized,light_v04_decimal_normalized_seed42,0.019,0.001,0.731,0.685,0.011,-0.008,ranked_by_valid_grouped
1,2,v06_keep_asy_if_short,light_v06_keep_asy_if_short_seed42,0.017,0.004,0.757,0.717,0.020,0.003,ranked_by_valid_grouped
2,3,v05_strip_asy,light_v05_strip_asy_seed42,0.015,0.003,0.756,0.716,0.025,0.010,ranked_by_valid_grouped


In [13]:
def resolve_test_file():
    if TEST_FILE:
        path = Path(TEST_FILE)
        if path.exists():
            return path
    for cand in [INPUT_ROOT / "test.jsonl", INPUT_ROOT / "test.json", Path("/kaggle/input/test.jsonl"), Path("/kaggle/input/test.json")]:
        if cand.exists():
            return cand
    return None


def load_model_with_adapter(adapter_dir):
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME_OR_PATH, local_files_only=LOCAL_FILES_ONLY)
    base = ensure_model_embeddings(base)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.config.pad_token_id = PAD_ID
    model.config.eos_token_id = EOS_ID
    return model.to("cuda" if torch.cuda.is_available() else "cpu")


def generate_submission_predictions(model, records, run_config, mode_cfg):
    preds = generate_predictions(model, records, run_config, "test", mode_cfg)
    out = []
    for pred in preds:
        out.append({
            "id": pred.get("id"),
            "query": pred.get("query"),
            "prompt": pred.get("prompt"),
            "generated_text": pred.get("generated_text"),
            "pred_final_answer_raw": pred.get("pred_final_answer_raw"),
            "pred_final_answer_normalized": pred.get("pred_final_answer_normalized"),
            "answer_extracted": pred.get("answer_extracted"),
            "format_has_anchor": pred.get("format_has_anchor"),
        })
    return out


if RUN_TEST_INFERENCE:
    if not SUBMISSION_RUN_ID:
        raise ValueError("Cần đặt SUBMISSION_RUN_ID trước khi chạy test inference.")
    test_path = resolve_test_file()
    if test_path is None:
        raise FileNotFoundError("Không tìm thấy test.json hoặc test.jsonl.")
    run_dir = OUTPUT_ROOT / "runs" / SUBMISSION_RUN_ID
    run_config = load_json_optional(run_dir / "run_config.json")
    if not run_config:
        raise FileNotFoundError(f"Không tìm thấy run_config.json cho {SUBMISSION_RUN_ID}")
    test_records = load_json_or_jsonl(test_path)
    normalized_test_records = []
    for idx, rec in enumerate(test_records):
        item = dict(rec)
        item.setdefault("id", rec.get("id", f"test_{idx:06d}"))
        item.setdefault("query", rec.get("query", rec.get("query_vi", "")))
        item.setdefault("response", "")
        item.setdefault("final_answer", "")
        normalized_test_records.append(item)
    model = load_model_with_adapter(run_dir / "adapter")
    mode_cfg = MODE_CONFIGS[run_config["run_mode"]]
    submission_predictions = generate_submission_predictions(model, normalized_test_records, run_config, mode_cfg)
    save_jsonl(submission_predictions, OUTPUT_ROOT / "submission_predictions.jsonl")
    pd.DataFrame([{"id": p["id"], "answer": p["pred_final_answer_normalized"]} for p in submission_predictions]).to_csv(OUTPUT_ROOT / "submission.csv", index=False, encoding="utf-8-sig")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Wrote submission outputs")
else:
    print("Skip test inference")

Skip test inference


In [14]:
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "kaggle_input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "run_modes_executed": [RUN_MODE],
    "variants_executed": [p.name for p in VARIANT_DIRS],
    "seed": SEED,
}
save_json(manifest, OUTPUT_ROOT / "notebook_run_manifest.json")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Done:", OUTPUT_ROOT)

{
  "created_at": "2026-05-18T07:58:08.251681+00:00",
  "kaggle_input_root": "/kaggle/input/datasets/phamanhtuanas/gpt2-math/preprocessing_experiments/preprocessing_experiments",
  "output_root": "/kaggle/working/variant_experiments",
  "run_modes_executed": [
    "light_ablation"
  ],
  "variants_executed": [
    "v04_decimal_normalized",
    "v05_strip_asy",
    "v06_keep_asy_if_short"
  ],
  "seed": 42
}
Done: /kaggle/working/variant_experiments
